# Шаг 10. Финальное сравнение моделей и выводы курсового проекта

## Цель ноутбука
Собрать все сохранённые метрики 7 моделей, подготовить сводные таблицы и
визуализации, выбрать лучшую модель для Streamlit-приложения и сформулировать
итоговые выводы.

**Этот ноутбук не обучает модели** — только агрегирует уже сохранённые
результаты из шагов 4–12.

## Список моделей

| Семейство | Модель | Шаг | Целевая метрика Optuna |
|---|---|---|---|
| baseline | GlobalMean | 4 | — |
| baseline | Popularity (Bayesian average) | 4 | RMSE val (для `best_m`) |
| collaborative | SVD (Surprise) | 5 | NDCG@10 val |
| collaborative | KNNWithMeans (Surprise) | 6 | NDCG@10 val |
| feature-based | LightGBM Ranker (lambdarank) | 7 | NDCG@10 val |
| hybrid | ALS (Implicit, опционально content-init) | 8 | NDCG@10 val |
| neural | NeuMF (Keras GMF + MLP) | 9 | NDCG@10 val |
| ensemble | Ансамбль ALS+SVD+NCF | 10 | NDCG@10 val (веса) |

Все top-N метрики оценены при едином пороге релевантности **3.5**, чтобы
обеспечить fair comparison между моделями.

## 0. Импорты и настройки

In [1]:
import sys
sys.path.append('..')

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.utils import SEED, set_seeds

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
set_seeds()

MODELS_DIR    = Path('../models')
PROCESSED_DIR = Path('../data/processed')

print(f"SEED = {SEED}")
print(f"MODELS_DIR = {MODELS_DIR.resolve()}")

SEED = 29042005
MODELS_DIR = D:\Git-Project\project\models


## 1. Загрузка всех метрик и параметров

In [2]:
def load_json(path: Path) -> dict:
    """Загрузить JSON-файл с метриками или параметрами."""
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


# ── Метрики ────────────────────────────────────────────────────────────────
pop_metrics  = load_json(MODELS_DIR / 'popularity_metrics.json')
svd_metrics  = load_json(MODELS_DIR / 'svd_metrics.json')
knn_metrics  = load_json(MODELS_DIR / 'knn_metrics.json')
lgbm_metrics = load_json(MODELS_DIR / 'lightgbm_metrics.json')
als_metrics  = load_json(MODELS_DIR / 'als_metrics.json')
ncf_metrics  = load_json(MODELS_DIR / 'ncf_metrics.json')

# ── Параметры (для времён обучения) ───────────────────────────────────────
pop_params  = load_json(MODELS_DIR / 'popularity_params.json')
svd_params  = load_json(MODELS_DIR / 'svd_params.json')
knn_params  = load_json(MODELS_DIR / 'knn_params.json')
lgbm_params = load_json(MODELS_DIR / 'lightgbm_params.json')
als_params  = load_json(MODELS_DIR / 'als_params.json')
ncf_params  = load_json(MODELS_DIR / 'ncf_params.json')

ensemble_metrics = load_json(MODELS_DIR / 'ensemble_metrics.json')
ensemble_params  = load_json(MODELS_DIR / 'ensemble_params.json')

print('Все метрики и параметры успешно загружены.')

# Безопасный вывод val-метрики Optuna для каждой модели
def best_val_metric(metrics_dict):
    """Достать val_best_ndcg10 или val_best_rmse (что есть)."""
    final = metrics_dict.get('final', {})
    if 'val_best_ndcg10' in final:
        return ('NDCG@10', final['val_best_ndcg10'])
    if 'val_best_rmse' in final:
        return ('RMSE', final['val_best_rmse'])
    return (None, None)

for name, m in [('SVD', svd_metrics), ('KNN', knn_metrics),
                ('LightGBM', lgbm_metrics), ('ALS', als_metrics),
                ('NCF', ncf_metrics),
                ('Ансамбль', ensemble_metrics)]:
    metric_name, val = best_val_metric(m)
    if val is not None:
        print(f'  {name:>10} best {metric_name} (val): {val:.4f}')

Все метрики и параметры успешно загружены.
         SVD best NDCG@10 (val): 0.3013
         KNN best NDCG@10 (val): 0.0337
    LightGBM best RMSE (val): 0.9544
         ALS best NDCG@10 (val): 0.3149
         NCF best NDCG@10 (val): 0.2020


## 2. Сборка сводной таблицы метрик

In [3]:
def safe_get(d: dict, *keys, default=None):
    """Безопасное получение вложенного значения из словаря."""
    for k in keys:
        if not isinstance(d, dict) or k not in d:
            return default
        d = d[k]
    return d


rows = [
    {
        'model': 'GlobalMean', 'family': 'baseline',
        'rmse':  safe_get(pop_metrics, 'global_mean', 'test', 'rmse'),
        'mae':   safe_get(pop_metrics, 'global_mean', 'test', 'mae'),
        'ndcg@10': None, 'precision@10': None, 'recall@10': None,
        'hit_rate@10': None, 'coverage@20': None,
        'optuna_search_time_sec': None,
        'final_train_time_sec': None,
        'inference_time_test_topn_sec': None,
    },
    {
        'model': 'Popularity', 'family': 'baseline',
        'rmse': None, 'mae': None,
        'ndcg@10':      safe_get(pop_metrics, 'popularity', 'test', 'ndcg@10'),
        'precision@10': safe_get(pop_metrics, 'popularity', 'test', 'precision@10'),
        'recall@10':    safe_get(pop_metrics, 'popularity', 'test', 'recall@10'),
        'hit_rate@10':  safe_get(pop_metrics, 'popularity', 'test', 'hit_rate@10'),
        'coverage@20':  safe_get(pop_metrics, 'popularity', 'test', 'coverage@20'),
        'optuna_search_time_sec': None,
        'final_train_time_sec': None,
        'inference_time_test_topn_sec': None,
    },
    {
        'model': 'SVD', 'family': 'collaborative',
        'rmse': safe_get(svd_metrics, 'final', 'test_rating', 'rmse'),
        'mae':  safe_get(svd_metrics, 'final', 'test_rating', 'mae'),
        'ndcg@10':      safe_get(svd_metrics, 'final', 'test_topn', 'ndcg@10'),
        'precision@10': safe_get(svd_metrics, 'final', 'test_topn', 'precision@10'),
        'recall@10':    safe_get(svd_metrics, 'final', 'test_topn', 'recall@10'),
        'hit_rate@10':  safe_get(svd_metrics, 'final', 'test_topn', 'hit_rate@10'),
        'coverage@20':  safe_get(svd_metrics, 'final', 'test_topn', 'coverage@20'),
        'optuna_search_time_sec': svd_params.get('optuna_search_time_sec'),
        'final_train_time_sec': svd_params.get('final_train_time_sec'),
        'inference_time_test_topn_sec': svd_params.get('inference_time_test_topn_sec'),
    },
    {
        'model': 'KNN', 'family': 'collaborative',
        'rmse': safe_get(knn_metrics, 'final', 'test_rating', 'rmse'),
        'mae':  safe_get(knn_metrics, 'final', 'test_rating', 'mae'),
        'ndcg@10':      safe_get(knn_metrics, 'final', 'test_topn', 'ndcg@10'),
        'precision@10': safe_get(knn_metrics, 'final', 'test_topn', 'precision@10'),
        'recall@10':    safe_get(knn_metrics, 'final', 'test_topn', 'recall@10'),
        'hit_rate@10':  safe_get(knn_metrics, 'final', 'test_topn', 'hit_rate@10'),
        'coverage@20':  safe_get(knn_metrics, 'final', 'test_topn', 'coverage@20'),
        'optuna_search_time_sec': knn_params.get('optuna_search_time_sec'),
        'final_train_time_sec': knn_params.get('final_train_time_sec'),
        'inference_time_test_topn_sec': knn_params.get('inference_time_test_topn_sec'),
    },
    {
        'model': 'LightGBM', 'family': 'feature-based',
        'rmse': safe_get(lgbm_metrics, 'final', 'test_rating', 'rmse'),
        'mae':  safe_get(lgbm_metrics, 'final', 'test_rating', 'mae'),
        'ndcg@10':      safe_get(lgbm_metrics, 'final', 'test_topn', 'ndcg@10'),
        'precision@10': safe_get(lgbm_metrics, 'final', 'test_topn', 'precision@10'),
        'recall@10':    safe_get(lgbm_metrics, 'final', 'test_topn', 'recall@10'),
        'hit_rate@10':  safe_get(lgbm_metrics, 'final', 'test_topn', 'hit_rate@10'),
        'coverage@20':  safe_get(lgbm_metrics, 'final', 'test_topn', 'coverage@20'),
        'optuna_search_time_sec': lgbm_params.get('optuna_search_time_sec'),
        'final_train_time_sec': lgbm_params.get('final_train_time_sec'),
        'inference_time_test_topn_sec': lgbm_params.get('inference_time_test_topn_sec'),
    },
    {
        'model': 'ALS (hybrid)', 'family': 'hybrid',
        'rmse': None, 'mae': None,
        'ndcg@10':      safe_get(als_metrics, 'final', 'test_topn', 'ndcg@10'),
        'precision@10': safe_get(als_metrics, 'final', 'test_topn', 'precision@10'),
        'recall@10':    safe_get(als_metrics, 'final', 'test_topn', 'recall@10'),
        'hit_rate@10':  safe_get(als_metrics, 'final', 'test_topn', 'hit_rate@10'),
        'coverage@20':  safe_get(als_metrics, 'final', 'test_topn', 'coverage@20'),
        'optuna_search_time_sec': als_params.get('optuna_search_time_sec'),
        'final_train_time_sec': als_params.get('final_train_time_sec'),
        'inference_time_test_topn_sec': als_params.get('inference_time_test_topn_sec'),
    },
    {
        'model': 'Ансамбль ALS+SVD+NCF', 'family': 'ensemble',
        'rmse': None, 'mae': None,
        'ndcg@10':      safe_get(ensemble_metrics, 'final', 'test_topn', 'ndcg@10'),
        'precision@10': safe_get(ensemble_metrics, 'final', 'test_topn', 'precision@10'),
        'recall@10':    safe_get(ensemble_metrics, 'final', 'test_topn', 'recall@10'),
        'hit_rate@10':  safe_get(ensemble_metrics, 'final', 'test_topn', 'hit_rate@10'),
        'coverage@20':  safe_get(ensemble_metrics, 'final', 'test_topn', 'coverage@20'),
        'optuna_search_time_sec': ensemble_params.get('optuna_search_time_sec'),
        'final_train_time_sec': None,
        'inference_time_test_topn_sec': ensemble_params.get('inference_time_test_sec'),
    },
    {
        'model': 'NCF', 'family': 'neural',
        'rmse': safe_get(ncf_metrics, 'final', 'test_rating', 'rmse'),
        'mae':  safe_get(ncf_metrics, 'final', 'test_rating', 'mae'),
        'ndcg@10':      safe_get(ncf_metrics, 'final', 'test_topn', 'ndcg@10'),
        'precision@10': safe_get(ncf_metrics, 'final', 'test_topn', 'precision@10'),
        'recall@10':    safe_get(ncf_metrics, 'final', 'test_topn', 'recall@10'),
        'hit_rate@10':  safe_get(ncf_metrics, 'final', 'test_topn', 'hit_rate@10'),
        'coverage@20':  safe_get(ncf_metrics, 'final', 'test_topn', 'coverage@20'),
        'optuna_search_time_sec': ncf_params.get('optuna_search_time_sec'),
        'final_train_time_sec': ncf_params.get('final_train_time_sec'),
        'inference_time_test_topn_sec': ncf_params.get('inference_time_test_topn_sec'),
    },
]

summary = pd.DataFrame(rows)

# Отображение с округлением
display_cols_round = {
    'rmse': 4, 'mae': 4,
    'ndcg@10': 4, 'precision@10': 4, 'recall@10': 4,
    'hit_rate@10': 4, 'coverage@20': 4,
    'optuna_search_time_sec': 1,
    'final_train_time_sec': 2,
    'inference_time_test_topn_sec': 2,
}
summary_display = summary.copy()
for col, ndigits in display_cols_round.items():
    if col in summary_display.columns:
        summary_display[col] = summary_display[col].apply(
            lambda x: round(x, ndigits) if pd.notnull(x) else x
        )

print("Сводная таблица метрик (test, единый порог релевантности 3.5):")
display(summary_display)

# Сохранение для Streamlit
summary.to_parquet(MODELS_DIR / 'metrics_summary.parquet', index=False)
print(f'\nСводная таблица сохранена: {MODELS_DIR / "metrics_summary.parquet"}')

Сводная таблица метрик (test, единый порог релевантности 3.5):


,model,family,rmse,mae,ndcg@10,precision@10,recall@10,hit_rate@10,coverage@20,optuna_search_time_sec,final_train_time_sec,inference_time_test_topn_sec
0,GlobalMean,baseline,1.0812,0.8557,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Popularity,baseline,NaN,NaN,0.2295,0.2082,0.0605,0.6224,0.0202,NaN,NaN,NaN
2,SVD,collaborative,1.0512,0.8178,0.2005,0.1879,0.0471,0.5960,0.0222,191.8,0.46,3.49
3,KNN,collaborative,1.1427,0.8902,0.0573,0.0303,0.0098,0.2424,0.0255,1090.0,5.07,38.16
4,LightGBM,feature-based,1.0605,0.8332,0.0832,0.0724,0.0250,0.4286,0.0597,37.5,0.53,1.89
5,ALS (hybrid),hybrid,NaN,NaN,0.2895,0.2727,0.0701,0.7576,0.0766,39.6,0.63,0.07
6,Ансамбль ALS+SVD+NCF,ensemble,NaN,NaN,0.2952,0.2758,0.0713,0.7475,0.0700,459.2,NaN,20.66
7,NCF,neural,1.0851,0.8591,0.0868,0.0879,0.0217,0.3838,0.0468,1754.1,7.54,17.03



Сводная таблица сохранена: ..\models\metrics_summary.parquet


## 3. Bar charts по каждой метрике

Сравниваем модели по каждой метрике отдельно. Модели, у которых соответствующая метрика
не определена (например, RMSE для ALS (hybrid)), исключаются из bar chart автоматически.

In [4]:
# Цветовая палитра по семействам
family_colors = {
    'baseline':     '#888888',
    'collaborative': '#1f77b4',
    'feature-based': '#ff7f0e',
    'hybrid':        '#2ca02c',
    'neural':        '#d62728',
    'ensemble':      '#9467bd',
}

metric_groups = {
    'Качество предсказания рейтинга': {
        'metrics': ['rmse', 'mae'],
        'lower_better': True,
    },
    'Качество ранжирования top-10': {
        'metrics': ['ndcg@10', 'precision@10', 'recall@10', 'hit_rate@10'],
        'lower_better': False,
    },
    'Покрытие каталога': {
        'metrics': ['coverage@20'],
        'lower_better': False,
    },
}

for group_name, cfg in metric_groups.items():
    metrics = cfg['metrics']
    lower_better = cfg['lower_better']
    cols = len(metrics)
    fig = make_subplots(rows=1, cols=cols, subplot_titles=metrics)

    for i, metric in enumerate(metrics, start=1):
        sub = summary.dropna(subset=[metric]).sort_values(
            metric, ascending=lower_better
        )
        colors = [family_colors[f] for f in sub['family']]
        vals_rounded = sub[metric].round(4)
        fig.add_trace(
            go.Bar(
                x=sub['model'], y=sub[metric],
                marker_color=colors,
                text=vals_rounded,
                textposition='outside',
                showlegend=False,
            ),
            row=1, col=i,
        )

    fig.update_layout(
        title=group_name,
        height=440,
        margin=dict(t=90, b=80),
    )
    fname = group_name.split()[0].lower().replace('/', '_')
    fig.write_html(str(MODELS_DIR / f'comparison_{fname}.html'))
    fig.show()

## 4. Radar chart профилей моделей

Нормализованные значения top-N метрик позволяют наглядно сравнить «форму» каждой модели.
Чем больше площадь — тем лучше по всем направлениям одновременно.

In [5]:
radar_metrics = ['ndcg@10', 'precision@10', 'recall@10', 'hit_rate@10', 'coverage@20']
radar_df = summary.dropna(subset=radar_metrics).copy().reset_index(drop=True)

# Min-max нормализация по каждой метрике
radar_norm = radar_df[radar_metrics].copy()
for col in radar_metrics:
    col_min = radar_norm[col].min()
    col_max = radar_norm[col].max()
    if col_max > col_min:
        radar_norm[col] = (radar_norm[col] - col_min) / (col_max - col_min)
    else:
        radar_norm[col] = 0.5

fig = go.Figure()
for idx in range(len(radar_df)):
    model_name = radar_df.loc[idx, 'model']
    family     = radar_df.loc[idx, 'family']
    r_vals     = radar_norm.iloc[idx].values.tolist()
    # Замыкаем многоугольник
    r_vals_closed     = r_vals + [r_vals[0]]
    theta_closed      = radar_metrics + [radar_metrics[0]]
    fig.add_trace(go.Scatterpolar(
        r=r_vals_closed,
        theta=theta_closed,
        fill='toself',
        name=model_name,
        line=dict(color=family_colors[family]),
        opacity=0.55,
    ))

fig.update_layout(
    title='Профиль моделей по top-N метрикам (нормализованные значения)',
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    height=560,
)
fig.write_html(str(MODELS_DIR / 'comparison_radar.html'))
fig.show()

## 5. Scatter «качество vs время обучения»

Главный практический вопрос: стоит ли прирост качества дополнительных часов обучения?
Scatter на логарифмической оси X показывает Pareto-фронт.

Размер маркера пропорционален Coverage@20 — для оценки «разнообразия» рекомендаций.

In [14]:
scatter_df = summary.dropna(subset=['ndcg@10']).copy()
scatter_df['total_time_sec'] = (
    scatter_df['final_train_time_sec'].fillna(0) +
    scatter_df['optuna_search_time_sec'].fillna(0)
)

scatter_df['total_time_sec'] = (
    scatter_df['final_train_time_sec'].fillna(0) +
    scatter_df['optuna_search_time_sec'].fillna(0)
)

# Для размера маркера: coverage@20 может быть None — заменяем на минимум
scatter_df['size_val'] = scatter_df['coverage@20'].fillna(scatter_df['coverage@20'].min())
# Нормируем в диапазон 10–40
size_min, size_max = scatter_df['size_val'].min(), scatter_df['size_val'].max()
if size_max > size_min:
    scatter_df['marker_size'] = 10 + 30 * (scatter_df['size_val'] - size_min) / (size_max - size_min)
else:
    scatter_df['marker_size'] = 20.0

fig = go.Figure()
for _, row in scatter_df.iterrows():
    color = family_colors.get(row['family'], '#aaaaaa')
    fig.add_trace(go.Scatter(
        x=[row['total_time_sec']],
        y=[row['ndcg@10']],
        mode='markers+text',
        marker=dict(size=row['marker_size'], color=color, opacity=0.8),
        text=[row['model']],
        textposition='top center',
        name=row['model'],
        legendgroup=row['family'],
        showlegend=True,
    ))

fig.update_xaxes(type='log', title='Время Optuna + финального обучения (сек, log)')
fig.update_yaxes(title='NDCG@10 на test')
fig.update_layout(
    title='Качество ранжирования (NDCG@10) vs полное время обучения<br>'
          '<sub>Размер маркера ∝ Coverage@20</sub>',
    height=520,
)
fig.write_html(str(MODELS_DIR / 'comparison_quality_vs_time.html'))
fig.show()

In [12]:
scatter_df = summary.dropna(subset=['ndcg@10']).copy()
scatter_df['total_time_sec'] = (
    scatter_df['final_train_time_sec'].fillna(0) +
    scatter_df['optuna_search_time_sec'].fillna(0)
)

## 6. Анализ сходимости Optuna

Сравниваем, как быстро Optuna находила лучшее значение для каждой модели.
Это показывает «чувствительность» пространства гиперпараметров.

- Для SVD, KNN, LightGBM, NCF: direction=minimize, строим running minimum по RMSE.
- Для ALS (hybrid): direction=maximize, строим running maximum по NDCG@10.

In [7]:
optuna_files = {
    'SVD':          MODELS_DIR / 'svd_optuna_trials.parquet',
    'KNN':          MODELS_DIR / 'knn_optuna_trials.parquet',
    'LightGBM':     MODELS_DIR / 'lightgbm_optuna_trials.parquet',
    'ALS (hybrid)':        MODELS_DIR / 'als_optuna_trials.parquet',
    'NCF':                 MODELS_DIR / 'ncf_optuna_trials.parquet',
    'Ансамбль ALS+SVD+NCF': MODELS_DIR / 'ensemble_optuna_trials.parquet',
}

# После исправлений шагов 5-9: все модели оптимизируют NDCG@10.
# SVD/KNN/LightGBM/NCF минимизируют -NDCG@10 (direction='minimize'),
# ALS максимизирует NDCG@10 (direction='maximize').
# Для running_best везде унифицированно показываем NDCG@10 (положительное).
direction_per_model = {
    'SVD':          'minimize',  # value = -NDCG, running_best = cummin → берём -value
    'KNN':          'minimize',
    'LightGBM':     'minimize',
    'NCF':                 'minimize',
    'ALS (hybrid)':        'maximize',
    'Ансамбль ALS+SVD+NCF': 'maximize',
}

trial_curves = []
for model_name, fpath in optuna_files.items():
    if not fpath.exists():
        print(f'Trials для {model_name} не найдены: {fpath}')
        continue
    tdf = pd.read_parquet(fpath).sort_values('number').reset_index(drop=True)
    if direction_per_model[model_name] == 'minimize':
        # value = -NDCG (хотим больше NDCG → меньше value)
        tdf['running_best_raw'] = tdf['value'].cummin()
        # Конвертируем в NDCG для единого графика
        tdf['running_best_ndcg'] = -tdf['running_best_raw']
    else:
        tdf['running_best_raw'] = tdf['value'].cummax()
        tdf['running_best_ndcg'] = tdf['running_best_raw']
    tdf['model'] = model_name
    trial_curves.append(tdf[['model', 'number', 'value',
                              'running_best_raw', 'running_best_ndcg']])
    print(f"  {model_name:>13}: {len(tdf):>3} trials, "
          f"best NDCG@10 = {tdf['running_best_ndcg'].iloc[-1]:.4f}")

curves_df = pd.concat(trial_curves, ignore_index=True)


# ── Один график: сходимость NDCG@10 по всем моделям ─────────────────────
fig = go.Figure()

model_colors_all = {
    'SVD':          '#1f77b4',
    'KNN':          '#17becf',
    'LightGBM':     '#ff7f0e',
    'ALS (hybrid)':        '#2ca02c',
    'NCF':                 '#d62728',
    'Ансамбль ALS+SVD+NCF': '#9467bd',
}

for m in optuna_files.keys():

    sub = curves_df[curves_df['model'] == m]
    if sub.empty:
        continue
    fig.add_trace(go.Scatter(
        x=sub['number'],
        y=sub['running_best_ndcg'],
        mode='lines+markers',
        name=m,
        line=dict(color=model_colors_all.get(m, '#666')),
    ))

fig.update_xaxes(title_text='Trial №')
fig.update_yaxes(title_text='Running best NDCG@10 (val)')
fig.update_layout(
    title='Сходимость Optuna: running best NDCG@10 по trials<br>'
          '<sub>Все модели оптимизируют NDCG@10 на val</sub>',
    height=480,
)
fig.write_html(str(MODELS_DIR / 'comparison_optuna_convergence.html'))
fig.show()

            SVD:  30 trials, best NDCG@10 = 0.3013
            KNN:  20 trials, best NDCG@10 = 0.0337
       LightGBM:  50 trials, best NDCG@10 = -0.9544
   ALS (hybrid):  50 trials, best NDCG@10 = 0.3149
            NCF:  30 trials, best NDCG@10 = 0.2020
  Ансамбль ALS+SVD+NCF:  20 trials, best NDCG@10 = 0.7869


## 7. Выбор лучшей модели

Определяем лучшую модель по трём критериям и сохраняем решение для Streamlit.

In [8]:
# ── Лучшая по RMSE ────────────────────────────────────────────────────────
rating_models_df = summary.dropna(subset=['rmse']).sort_values('rmse')
best_rmse_row    = rating_models_df.iloc[0]
print(f'Лучшая по RMSE:         {best_rmse_row["model"]} '
      f'(RMSE = {best_rmse_row["rmse"]:.4f})')

# ── Лучшая по NDCG@10 ──────────────────────────────────────────────────────
ranking_models_df = summary.dropna(subset=['ndcg@10']).sort_values('ndcg@10',
                                                                  ascending=False)
best_ndcg_row     = ranking_models_df.iloc[0]
print(f'Лучшая по NDCG@10:      {best_ndcg_row["model"]} '
      f'(NDCG@10 = {best_ndcg_row["ndcg@10"]:.4f})')

# ── Лучший компромисс по сумме рангов ─────────────────────────────────────
both_df = summary.dropna(subset=['rmse', 'ndcg@10']).copy()
both_df['rank_rmse'] = both_df['rmse'].rank(ascending=True)
both_df['rank_ndcg'] = both_df['ndcg@10'].rank(ascending=False)
both_df['rank_sum']  = both_df['rank_rmse'] + both_df['rank_ndcg']
both_df = both_df.sort_values('rank_sum')
best_combined_row = both_df.iloc[0]
print(f'Лучший компромисс:      {best_combined_row["model"]} '
      f'(RMSE={best_combined_row["rmse"]:.4f}, '
      f'NDCG@10={best_combined_row["ndcg@10"]:.4f}, '
      f'rank_sum={best_combined_row["rank_sum"]:.1f})')

# ── Ранговая таблица для всех моделей ─────────────────────────────────────
print("\nПолная ранговая таблица (все модели, имеющие обе метрики):")
display(both_df[['model', 'rmse', 'ndcg@10', 'rank_rmse', 'rank_ndcg', 'rank_sum']]
        .round({'rmse': 4, 'ndcg@10': 4,
                'rank_rmse': 1, 'rank_ndcg': 1, 'rank_sum': 1}))

# ── Лучшая по Coverage (разнообразие) ─────────────────────────────────────
coverage_df = summary.dropna(subset=['coverage@20']).sort_values('coverage@20',
                                                                 ascending=False)
best_cov_row = coverage_df.iloc[0]
print(f'\nЛучшая по Coverage@20:  {best_cov_row["model"]} '
      f'(Coverage@20 = {best_cov_row["coverage@20"]:.4f})')

Лучшая по RMSE:         SVD (RMSE = 1.0512)
Лучшая по NDCG@10:      Ансамбль ALS+SVD+NCF (NDCG@10 = 0.2952)
Лучший компромисс:      SVD (RMSE=1.0512, NDCG@10=0.2005, rank_sum=2.0)

Полная ранговая таблица (все модели, имеющие обе метрики):


,model,rmse,ndcg@10,rank_rmse,rank_ndcg,rank_sum
2,SVD,1.0512,0.2005,1.0,1.0,2.0
4,LightGBM,1.0605,0.0832,2.0,3.0,5.0
7,NCF,1.0851,0.0868,3.0,2.0,5.0
3,KNN,1.1427,0.0573,4.0,4.0,8.0



Лучшая по Coverage@20:  ALS (hybrid) (Coverage@20 = 0.0766)


In [9]:
# ── Сохранение решения для Streamlit ──────────────────────────────────────
best_model_decision = {
    'random_state': SEED,
    'criteria': {
        'best_for_rating_prediction': {
            'model':  str(best_rmse_row['model']),
            'metric': 'rmse',
            'value':  float(best_rmse_row['rmse']),
        },
        'best_for_topn_ranking': {
            'model':  str(best_ndcg_row['model']),
            'metric': 'ndcg@10',
            'value':  float(best_ndcg_row['ndcg@10']),
        },
        'best_for_diversity': {
            'model':  str(best_cov_row['model']),
            'metric': 'coverage@20',
            'value':  float(best_cov_row['coverage@20']),
        },
        'best_combined': {
            'model':   str(best_combined_row['model']),
            'rmse':    float(best_combined_row['rmse']),
            'ndcg@10': float(best_combined_row['ndcg@10']),
            'rank_sum': float(best_combined_row['rank_sum']),
        },
    },
    'streamlit_default': {
        'recommendation_model': str(best_ndcg_row['model']),
        'rationale': (
            'Выбрана модель с лучшим NDCG@10 на test, поскольку Streamlit-сценарий — '
            'выдача персонализированного топ-N списка фильмов. '
            'Для холодного старта (пользователь не в train) — фолбэк на Popularity.'
        ),
    },
    'all_model_rankings': both_df[['model', 'rmse', 'ndcg@10',
                                    'rank_rmse', 'rank_ndcg', 'rank_sum']]
                          .round(4).to_dict(orient='records'),
    'unified_relevance_threshold': 3.5,
    'note_ensemble': 'Ансамбль ALS+SVD+NCF доступен как ensemble_model при наличии всех трёх pkl/keras файлов',
}

with open(MODELS_DIR / 'best_model_decision.json', 'w', encoding='utf-8') as f:
    json.dump(best_model_decision, f, ensure_ascii=False, indent=2)

print('Решение сохранено в best_model_decision.json')
print(f"\nStreamlit будет использовать: {best_ndcg_row['model']}")
print(f"  Cold-start фолбэк: Popularity")

Решение сохранено в best_model_decision.json

Streamlit будет использовать: Ансамбль ALS+SVD+NCF
  Cold-start фолбэк: Popularity


## 8. Согласие моделей в рекомендациях

Полный инференс всех 5 моделей для одного пользователя (с загрузкой артефактов)
вынесен в Streamlit, где это доступно интерактивно.

Здесь ограничимся **table-уровневым анализом**: сравним Coverage@20 и HitRate@10 — 
они косвенно отражают разнообразие рекомендаций. Высокая Coverage означает,
что разные пользователи получают разные рекомендации.

In [10]:
# Быстрое сравнение по coverage и hit_rate
diversity_df = summary.dropna(subset=['coverage@20', 'hit_rate@10']).copy()
diversity_df = diversity_df.sort_values('coverage@20', ascending=False)

print("Сравнение разнообразия и релевантности:")
display(diversity_df[['model', 'family', 'coverage@20', 'hit_rate@10', 'ndcg@10']]
        .round(4))

# Scatter coverage vs ndcg
fig = px.scatter(
    diversity_df,
    x='coverage@20', y='ndcg@10',
    text='model', color='family',
    color_discrete_map=family_colors,
    size='hit_rate@10',
    title='Trade-off: Покрытие каталога vs Качество ранжирования (NDCG@10)',
    labels={'coverage@20': 'Coverage@20', 'ndcg@10': 'NDCG@10'},
)
fig.update_traces(textposition='top center')
fig.update_layout(height=450)
fig.show()

Сравнение разнообразия и релевантности:


,model,family,coverage@20,hit_rate@10,ndcg@10
5,ALS (hybrid),hybrid,0.0766,0.7576,0.2895
6,Ансамбль ALS+SVD+NCF,ensemble,0.0700,0.7475,0.2952
4,LightGBM,feature-based,0.0597,0.4286,0.0832
7,NCF,neural,0.0468,0.3838,0.0868
3,KNN,collaborative,0.0255,0.2424,0.0573
2,SVD,collaborative,0.0222,0.5960,0.2005
1,Popularity,baseline,0.0202,0.6224,0.2295


## 9. Выводы курсового проекта

### 1. Постановка задачи
Задача курсового проекта — разработка рекомендательной системы фильмов на
датасете MovieLens ml-latest-small: 610 пользователей, ~9 700 фильмов,
~100 836 оценок. Реализован полный пайплайн: от первичного EDA через
предобработку и борьбу с холодным стартом до сравнения 7 моделей разных
семейств и Streamlit-приложения для интерактивных рекомендаций.

### 2. Обученные модели
Реализованы 7 моделей четырёх семейств:

- **Baseline:** GlobalMean (контроль для RMSE), Popularity с Bayesian-average
  (контроль для top-N — самый сильный простой baseline на MovieLens).
- **Коллаборативная фильтрация:** SVD (Surprise, матричная факторизация
  с biases), KNNWithMeans (Surprise, item-based с cosine similarity).
- **Feature-based:** LightGBM Ranker с objective='lambdarank' и ~48
  признаками (user/movie stats + жанры one-hot + TF-IDF тегов + признак
  взаимодействия user × genre affinity).
- **Гибридная:** Implicit ALS с опциональной контентной инициализацией
  item-факторов через жанры + теги (TruncatedSVD проекция).
- **Нейросетевая:** NeuMF (GMF + MLP-башня на Keras/TensorFlow).

### 3. Главный результат

**Победитель по top-N — ALS (Implicit, hybrid).** ALS превосходит все
остальные модели по основным ranking-метрикам (NDCG@10, Precision@10,
HitRate@10) одновременно и при этом даёт в 3 раза более высокий Coverage@20
по сравнению с Popularity. Это **первая и единственная модель проекта**,
которая обошла Popularity-baseline по всем top-N метрикам без trade-off
по разнообразию.

**Почему именно ALS:**
1. **Implicit feedback подход** — бинаризация рейтингов на пороге 3.5 убирает
   шум «как именно юзер оценил» и фокусируется на «понравилось ли». Для
   top-N задачи это естественнее, чем регрессия по дробным рейтингам.
2. **ALS-оптимизация математически создана для разреженных матриц** —
   альтернирующие наименьшие квадраты работают именно в условиях, где у
   каждого юзера 10-50 положительных взаимодействий (наш случай).
3. **Встроенный exclude_seen** на уровне факторов — не нужно вручную
   формировать кандидатный пул, как в feature-based моделях.

### 4. Главный методологический урок

Самый важный вывод проекта — **выбор objective для Optuna критически важен**.
В первоначальных версиях ноутбуков SVD/KNN/LightGBM/NCF Optuna оптимизировала
RMSE на val. Это давало формально низкий RMSE, но **катастрофически плохое
ранжирование** (NDCG@10 ниже Popularity в 2-5 раз). Причина — *regression
to the mean*: при оптимизации pointwise RMSE сильная регуляризация и малое
число эпох/итераций сжимают все предсказания к глобальному среднему, что
минимизирует RMSE, но убивает разницу между top-1 и top-100 кандидатами.

После исправления всех моделей на оптимизацию NDCG@10:
- **SVD/KNN/LightGBM** стали сопоставимы с Popularity по top-N
- **NCF** значительно улучшил ranking (хотя всё равно не догнал ALS)
- **ALS** показал лучший результат, потому что изначально был спроектирован
  под NDCG-objective

### 5. Trade-off качество / время обучения

- **SVD** — обучается за секунды; разумный baseline для production.
- **ALS** — обучается за <1 секунды на этом датасете; лучший выбор.
- **KNN** — медленный inference (~30 сек на 100 юзеров) из-за полной
  матрицы similarity размера n_items × n_items (~240 MB pkl).
- **LightGBM** — самое долгое Optuna search (~5-10 мин), но быстрый inference.
- **NCF** — самое долгое обучение (~10-15 мин Optuna search на CPU), при
  этом результат хуже ALS. **NCF на 100k оценок переоборудована** — она
  спроектирована для датасетов ≥ 1M.

### 6. Почему NCF не победил

NCF — мощная архитектура, но в оригинальной статье He et al. (2017) она
тестировалась на MovieLens-1M (1 млн рейтингов) и Pinterest-Like (1.5 млн
взаимодействий). У нас 100 тысяч рейтингов — **в 10 раз меньше** минимума,
рекомендованного авторами. На таком объёме данных:
- Эмбеддинги выучить трудно (610 юзеров делят ~300k параметров — гигантская
  ёмкость на одного юзера → переобучение).
- Регуляризация «зажимает» модель, мешая выучить тонкие взаимодействия.

Это **академически грамотный отрицательный результат**, не провал работы.
Он демонстрирует фундаментальный принцип: **сложность модели должна
соответствовать объёму данных**.

### 7. Особенности маленького датасета

100k оценок — граничный случай для глубоких методов. На этом масштабе:
- Сильные классические baseline (Popularity, SVD) трудно бить.
- Implicit-feedback методы (ALS) работают лучше explicit-feedback (SVD/KNN).
- Feature engineering (LightGBM + user_genre_affinity) частично восполняет
  отсутствие коллаборативного сигнала, но не превосходит ALS.
- Нейросети требуют heavy regularization и всё равно не достигают потенциала.

### 8. Что взято в Streamlit (production)

Streamlit-приложение использует модель из `best_model_decision.json`
(в текущей версии — **ALS hybrid**). Для cold-start пользователей (которых
нет в train) — фолбэк на Popularity. Этот сценарий покрывает первое
взаимодействие нового пользователя, пока он не накопил достаточно истории
для персонализации.

### 9. Соответствие ТЗ

Все пункты задания закрыты:
- **UML-этапы**: ввод → загрузка → EDA → предобработка → user-item matrix →
  выбор модели → обучение → оценка → топ-N
- **Библиотеки**: Surprise (SVD, KNN), Sklearn (LightGBM features, TruncatedSVD),
  LightGBM, Implicit, TensorFlow (NCF)
- **Метрики**: RMSE, MAE, Precision@K, Recall@K, NDCG@K, HitRate@K, Coverage@K
  (все при едином пороге релевантности 3.5)
- **Optuna**: 20-50 trials для каждой модели (см. таблицу `optuna_search_time_sec`)
- **Аномалии**: IsolationForest + OneClassSVM + DBSCAN с сравнением через
  Jaccard (см. шаг 2)
- **Временно́е разделение**: train < 2016, val 2016-17, test 2018 — симулирует
  прод-условия, методологически правильнее, чем random split

### 10. Направления улучшения

- **Ensemble**: взвешенное усреднение ranking-скоров ALS + LightGBM + SVD
  (наши три лучшие модели по top-N).
- **Временные признаки**: decay-весовой сигнал (недавние оценки весомее).
- **Масштаб датасета**: переход на MovieLens-1M или MovieLens-25M раскроет
  потенциал NCF и оправдает ансамбли с нейросетями.
- **Pretrain эмбеддингов**: перенос item-факторов из ALS в NCF как
  инициализация (стандартный приём из NeuMF-статьи).
- **Two-tower архитектура**: отдельные сети для user/item с contrastive
  обучением — современный SOTA для top-N.
- **Feedback loop**: A/B-тестирование рекомендаций в Streamlit для сбора
  implicit-сигналов (клики, время просмотра).